# Ep. 04 — Tool calling

Until now we have been building an **assistant**: a chat that answers questions with text. This episode is the turning point — we start building an **agent**: a system that can **act** by calling tools (functions your code runs) and using their results.

Alone, the model only generates text from the prompt. With **tools**, it can ask your code to run a function (weather, order lookup, database) and use the result in the final answer.

## The tool loop (harness)

The LLM sits at the center. It may emit a tool call; your harness runs the tool and returns the result; the loop continues until the model stops calling tools and produces the final answer.

```mermaid
flowchart LR
  U[User] --> LLM
  LLM -->|tool call| T[Tool]
  T -->|result| LLM
  LLM -->|final answer| A[Answer]
```


## Limitation — without tools, the model only "talks"

With no tools registered, the assistant **does not fetch external data**. Ask for the temperature now: it invents, refuses, or relies on static training knowledge — there is no weather API in the loop.


In [ ]:
import os
import json
import urllib.request

API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
API_URL = "https://api.openai.com/v1/chat/completions"

def ask(messages):
    """messages: string OR list of dicts {role, content}."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if not API_KEY:
        raise SystemExit("Set OPENAI_API_KEY and run again.")
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"]

# No tools: the model only has the prompt — no live weather.
question = "What is the temperature right now in São Paulo?"
answer = ask(question)
print(answer)


## Define a tool (OpenAI-style JSON Schema)

Each tool is a `type: function` object with `name`, `description`, and `parameters` (JSON Schema). The model **decides** whether to call and with which arguments; **your code** executes.


In [ ]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Returns current weather (simulated local data) for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name, e.g. São Paulo",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# Deterministic data — the tool body runs offline; the LLM still needs the API.
WEATHER = {
    "são paulo": {"temp_c": 23, "condition": "partly cloudy", "humidity": 68},
    "sao paulo": {"temp_c": 23, "condition": "partly cloudy", "humidity": 68},
    "lisboa": {"temp_c": 19, "condition": "sunny", "humidity": 55},
    "madrid": {"temp_c": 27, "condition": "clear sky", "humidity": 40},
}

def get_weather(city: str) -> str:
    key = city.strip().lower()
    data = WEATHER.get(key)
    if not data:
        data = {
            "temp_c": None,
            "condition": "unknown",
            "humidity": None,
            "note": f"no data for {city}",
        }
    return json.dumps({"city": city, **data}, ensure_ascii=False)

FUNCTIONS = {
    "get_weather": get_weather,
}

print("TOOLS:", json.dumps(TOOLS, indent=2, ensure_ascii=False))
print("demo get_weather:", get_weather("São Paulo"))


## Register tools and verify `tool_calls`

This loop is what turns the assistant into an **agent**: the model acts through tools until it stops and returns the final answer.

Send `tools=[...]` and `tool_choice="auto"` in the body. After each response:

1. Inspect `message.tool_calls`.
2. **Only if** the list is non-empty → invoke local functions, append `role: tool`, call again.
3. If there are no `tool_calls` → use the assistant `content` (do not invoke anything).


In [ ]:
def chat_completion(messages, tools=None):
    """One API call; returns the full message object (may include tool_calls)."""
    if not API_KEY:
        raise SystemExit("Set OPENAI_API_KEY and run again.")
    payload = {"model": MODEL, "messages": messages}
    if tools is not None:
        payload["tools"] = tools
        payload["tool_choice"] = "auto"
    body = json.dumps(payload).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]

def ask_with_tools(messages, tools=TOOLS, functions=FUNCTIONS, max_rounds=3):
    """
    Loop with explicit verification:
      1) call the API with tools registered
      2) inspect message.tool_calls
      3) ONLY if tool_calls are present → execute, append role=tool, call again
      4) if none → use normal assistant content (do not invoke anything)
    """
    messages = list(messages)
    for round_i in range(max_rounds):
        msg = chat_completion(messages, tools=tools)

        # --- explicit verification ---
        tool_calls = msg.get("tool_calls") or []
        has_tool_calls = isinstance(tool_calls, list) and len(tool_calls) > 0
        print(f"[round {round_i}] tool_calls present? {has_tool_calls} (n={len(tool_calls)})")

        assistant = {"role": "assistant", "content": msg.get("content")}
        if has_tool_calls:
            assistant["tool_calls"] = tool_calls
        messages.append(assistant)

        if not has_tool_calls:
            # No tool_calls: do not invoke functions; return assistant text.
            content = msg.get("content") or ""
            print("No tool requested — using assistant content.")
            return content

        # Only reaches here when tool_calls is a non-empty list.
        for tc in tool_calls:
            name = tc["function"]["name"]
            args = json.loads(tc["function"]["arguments"] or "{}")
            print(f"→ invoke: {name}({args})")
            result = functions[name](**args)
            print(f"← result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc["id"],
                "content": result,
            })
        # next iteration: second (or Nth) completion with the results

    raise RuntimeError("max_rounds reached without a final answer")

messages = [
    {
        "role": "system",
        "content": "You are an assistant. Use get_weather when you need current weather.",
    },
    {"role": "user", "content": "What is the temperature right now in São Paulo?"},
]
final = ask_with_tools(messages)
print("---")
print(final)


## Takeaways

1. Without tools, the LLM only completes text from context — no side effects.
2. Tools = schema on the request + local function + `role: tool` message with the result.
3. Always **verify** `tool_calls` before invoking; if empty, return normal `content`.
